# Лабораторная работа: Тематическое моделирование с BigARTM

**Цель:** Построить тематическую модель коллекции новостных статей с применением аддитивной регуляризации.

**Данные:** Новостные статьи с mk.ru (датасет MLSUM, русский язык)

## 0. Импорты и проверка окружения

In [1]:
import os
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

import artm

print(f'BigARTM version: {artm.version()}')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

ModuleNotFoundError: No module named 'artm'

## 1. Загрузка данных

In [ ]:
# Загрузка обучающей и тестовой выборок
# Если файлы уже скачаны, укажите путь к ним. Иначе — скачайте вручную по ссылкам из задания.

TRAIN_PATH = 'train_0000.parquet'  # переименуйте скачанный файл
TEST_PATH  = 'test_0000.parquet'

# Скачивание (если нет файлов на диске)
import urllib.request

TRAIN_URL = 'https://huggingface.co/datasets/mlsum/resolve/refs%2Fconvert%2Fparquet/ru/train/0000.parquet?download=true'
TEST_URL  = 'https://huggingface.co/datasets/mlsum/resolve/refs%2Fconvert%2Fparquet/ru/test/0000.parquet?download=true'

if not os.path.exists(TRAIN_PATH):
    print('Скачиваем обучающую выборку...')
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)
    print('Готово!')

if not os.path.exists(TEST_PATH):
    print('Скачиваем тестовую выборку...')
    urllib.request.urlretrieve(TEST_URL, TEST_PATH)
    print('Готово!')

df_train = pd.read_parquet(TRAIN_PATH, engine='pyarrow')
df_test  = pd.read_parquet(TEST_PATH,  engine='pyarrow')

print(f'Обучающая выборка: {df_train.shape}')
print(f'Тестовая выборка:  {df_test.shape}')
df_train.head(3)

In [ ]:
# Смотрим на структуру данных
print('Столбцы:', df_train.columns.tolist())
print('\nПример текста:')
print(df_train['text'].iloc[0][:500])

## 2. Предварительная обработка текста

In [ ]:
# Установка необходимых пакетов для лемматизации
# Запустите один раз: pip install pymorphy2 nltk

import pymorphy2
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

morph = pymorphy2.MorphAnalyzer()
STOP_WORDS = set(stopwords.words('russian'))

# Добавляем дополнительные стоп-слова
EXTRA_STOP = {
    'это', 'весь', 'свой', 'который', 'также', 'ещё', 'уже', 'очень',
    'более', 'менее', 'при', 'про', 'быть', 'год', 'лет', 'человек'
}
STOP_WORDS.update(EXTRA_STOP)

def preprocess_text(text: str) -> str:
    """Очистка, токенизация и лемматизация текста."""
    if not isinstance(text, str):
        return ''
    # Удаляем не-кириллические символы и цифры
    text = re.sub(r'[^а-яёА-ЯЁ\s]', ' ', text)
    text = text.lower()
    tokens = text.split()
    # Лемматизация + фильтрация
    lemmas = [
        morph.parse(tok)[0].normal_form
        for tok in tokens
        if len(tok) > 2 and tok not in STOP_WORDS
    ]
    # Убираем стоп-слова после лемматизации
    lemmas = [l for l in lemmas if l not in STOP_WORDS and len(l) > 2]
    return ' '.join(lemmas)

print('Тестируем предобработку:')
sample = df_train['text'].iloc[0][:200]
print('До: ', sample)
print('После:', preprocess_text(sample))

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

# Ограничим выборку для ускорения (можно убрать ограничение)
N_TRAIN = 10000   # увеличьте до df_train.shape[0] для полного датасета
N_TEST  = 2000

print(f'Обрабатываем {N_TRAIN} обучающих и {N_TEST} тестовых документов...')

df_train_small = df_train.head(N_TRAIN).copy()
df_test_small  = df_test.head(N_TEST).copy()

df_train_small['processed'] = df_train_small['text'].progress_apply(preprocess_text)
df_test_small['processed']  = df_test_small['text'].progress_apply(preprocess_text)

# Убираем пустые тексты
df_train_small = df_train_small[df_train_small['processed'].str.strip() != ''].reset_index(drop=True)
df_test_small  = df_test_small[df_test_small['processed'].str.strip() != ''].reset_index(drop=True)

print(f'После фильтрации — train: {len(df_train_small)}, test: {len(df_test_small)}')

## 3. Векторизация и подготовка батчей для BigARTM

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer: мешок слов
vectorizer = CountVectorizer(
    max_df=0.85,    # игнорируем слова, встречающиеся в >85% документов
    min_df=5,       # игнорируем слова, встречающиеся менее чем в 5 документах
    max_features=20000,
    ngram_range=(1, 1)
)

# Обучаем на train, трансформируем train и test
X_train = vectorizer.fit_transform(df_train_small['processed'])
X_test  = vectorizer.transform(df_test_small['processed'])

vocab = vectorizer.get_feature_names_out()

print(f'Размер матрицы train: {X_train.shape}')
print(f'Размер матрицы test:  {X_test.shape}')
print(f'Размер словаря: {len(vocab)}')

In [ ]:
import scipy.sparse as sp

# ---- Вспомогательная функция: sparse matrix -> BigARTM batches ----

def sparse_to_batches(X, vocab, batch_dir, batch_size=1000):
    """
    Конвертирует разреженную матрицу (документы x слова) в набор батчей BigARTM.
    Также создаёт dictionary-файл.
    """
    os.makedirs(batch_dir, exist_ok=True)

    # Создаём словарь BigARTM
    dictionary = artm.Dictionary()
    
    # Запишем батчи
    n_docs = X.shape[0]
    batch_paths = []

    for batch_idx, start in enumerate(range(0, n_docs, batch_size)):
        end = min(start + batch_size, n_docs)
        batch = artm.messages.Batch()
        batch.id = str(batch_idx)

        # Добавляем токены словаря в батч
        for token in vocab:
            batch.token.append(token)
            batch.class_id.append('@default_class')

        X_sub = X[start:end]
        cx = X_sub.tocoo()

        # Группируем по документам
        doc_items = {}
        for doc_local, word_idx, count in zip(cx.row, cx.col, cx.data):
            doc_items.setdefault(doc_local, []).append((word_idx, int(count)))

        for doc_local in range(end - start):
            item = batch.item.add()
            item.id = start + doc_local
            field = item.field.add()
            if doc_local in doc_items:
                for word_idx, count in doc_items[doc_local]:
                    field.token_id.append(word_idx)
                    field.token_count.append(count)

        batch_path = os.path.join(batch_dir, f'batch_{batch_idx:04d}.batch')
        with open(batch_path, 'wb') as f:
            f.write(batch.SerializeToString())
        batch_paths.append(batch_path)

    print(f'Записано {len(batch_paths)} батч(ей) в {batch_dir}')
    return batch_paths


# Создаём батчи
TRAIN_BATCH_DIR = 'batches_train'
TEST_BATCH_DIR  = 'batches_test'

train_batches = sparse_to_batches(X_train, vocab, TRAIN_BATCH_DIR, batch_size=500)
test_batches  = sparse_to_batches(X_test,  vocab, TEST_BATCH_DIR,  batch_size=500)

In [ ]:
# Создаём объект BatchVectorizer для train и test
batch_vectorizer_train = artm.BatchVectorizer(
    data_path=TRAIN_BATCH_DIR,
    data_format='batches'
)

batch_vectorizer_test = artm.BatchVectorizer(
    data_path=TEST_BATCH_DIR,
    data_format='batches'
)

# Создаём словарь из обучающего набора
dictionary = artm.Dictionary()
dictionary.gather(data_path=TRAIN_BATCH_DIR)

print('BatchVectorizer и Dictionary готовы.')
print(f'Всего токенов в словаре: {dictionary.get_tokens_count()}')

## 4. Обучение модели LDA (BigARTM) и график перплексии

In [ ]:
def build_lda_model(n_topics, dictionary, batch_vectorizer_train, n_passes=10, seed=42):
    """
    Создаёт и обучает LDA-модель через BigARTM (ARTM с симметричными Dirichlet-приорами).
    Возвращает модель и список значений перплексии по итерациям.
    """
    model = artm.ARTM(
        num_topics=n_topics,
        dictionary=dictionary,
        num_document_passes=5,     # число проходов E-шага внутри документа
        seed=seed,
        cache_theta=True
    )

    # Метрики
    model.scores.add(artm.PerplexityScore(
        name='perplexity',
        dictionary=dictionary
    ))
    model.scores.add(artm.SparsityPhiScore(name='sparsity_phi'))
    model.scores.add(artm.SparsityThetaScore(name='sparsity_theta'))
    model.scores.add(artm.TopTokensScore(name='top_tokens', num_tokens=10))

    # LDA-регуляризаторы (симметричный Dirichlet)
    model.regularizers.add(artm.DirichletPhiRegularizer(
        name='dirichlet_phi', tau=-0.1
    ))
    model.regularizers.add(artm.DirichletThetaRegularizer(
        name='dirichlet_theta', tau=-0.1
    ))

    perplexity_values = []

    for i in range(n_passes):
        model.fit_offline(batch_vectorizer=batch_vectorizer_train, num_collection_passes=1)
        perp = model.score_tracker['perplexity'].last_value
        perplexity_values.append(perp)
        print(f'  Итерация {i+1:2d}/{n_passes} | Перплексия: {perp:.1f}')

    return model, perplexity_values


print('Обучаем базовую LDA с 10 темами...')
model_lda, perp_lda = build_lda_model(
    n_topics=10,
    dictionary=dictionary,
    batch_vectorizer_train=batch_vectorizer_train,
    n_passes=15
)

In [ ]:
# График перплексии для базовой LDA
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(perp_lda) + 1), perp_lda, marker='o', linewidth=2, color='steelblue')
ax.set_xlabel('Итерация обучения', fontsize=13)
ax.set_ylabel('Перплексия', fontsize=13)
ax.set_title('LDA (10 тем) — изменение перплексии в ходе обучения', fontsize=14)
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('lda_perplexity_training.png', dpi=150)
plt.show()
print(f'Финальная перплексия (train): {perp_lda[-1]:.1f}')

## 5. Подбор оптимального числа тем с помощью перплексии

In [ ]:
topic_variants = [5, 10, 15, 20, 30]
n_passes_selection = 10  # для ускорения подбора

results = {}  # n_topics -> (final_perplexity, perplexity_curve)

for n_t in topic_variants:
    print(f'\n>>> Обучаем LDA с {n_t} темами...')
    m, perp_curve = build_lda_model(
        n_topics=n_t,
        dictionary=dictionary,
        batch_vectorizer_train=batch_vectorizer_train,
        n_passes=n_passes_selection
    )
    results[n_t] = (perp_curve[-1], perp_curve, m)

In [ ]:
# График сравнения перплексий для разного числа тем
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Левый: кривые обучения
colors = plt.cm.tab10(np.linspace(0, 1, len(topic_variants)))
for (n_t, (final_p, curve, _)), color in zip(results.items(), colors):
    axes[0].plot(range(1, len(curve)+1), curve, marker='o', label=f'{n_t} тем', color=color, linewidth=2)
axes[0].set_xlabel('Итерация', fontsize=12)
axes[0].set_ylabel('Перплексия', fontsize=12)
axes[0].set_title('Кривые обучения при разном числе тем', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Правый: финальная перплексия vs число тем
n_topics_list = list(results.keys())
final_perps   = [results[n][0] for n in n_topics_list]
axes[1].plot(n_topics_list, final_perps, marker='s', color='tomato', linewidth=2, markersize=8)
axes[1].set_xlabel('Число тем', fontsize=12)
axes[1].set_ylabel('Финальная перплексия', fontsize=12)
axes[1].set_title('Перплексия vs Число тем (метод «локтя»)', fontsize=13)
axes[1].grid(True, linestyle='--', alpha=0.6)
for n_t, fp in zip(n_topics_list, final_perps):
    axes[1].annotate(str(n_t), (n_t, fp), textcoords='offset points', xytext=(0, 10), fontsize=10)

plt.tight_layout()
plt.savefig('topic_selection.png', dpi=150)
plt.show()

best_n = n_topics_list[np.argmin(final_perps)]
print(f'\nОптимальное число тем по минимуму перплексии: {best_n}')
print('\nПерплексия по вариантам:')
for n_t, fp in zip(n_topics_list, final_perps):
    marker = ' <-- оптимум' if n_t == best_n else ''
    print(f'  {n_t:3d} тем: {fp:.1f}{marker}')

## 6. Определение тем тестовой выборки и перплексия теста

In [ ]:
# Берём лучшую модель (или дообучаем на оптимальном n_topics)
best_model_lda = results[best_n][2]

# Добавляем PerplexityScore на тестовых данных
best_model_lda.scores.add(artm.PerplexityScore(
    name='perplexity_test',
    dictionary=dictionary
))

# Трансформируем тестовые документы (получаем theta — тема×документ)
theta_test = best_model_lda.transform(batch_vectorizer=batch_vectorizer_test)

print(f'Матрица theta (тема×документ) для теста: {theta_test.shape}')

# Рассчитываем перплексию на тестовой выборке
test_perp = best_model_lda.score_tracker['perplexity_test'].last_value
print(f'\nПерплексия на тестовой выборке (LDA, {best_n} тем): {test_perp:.1f}')

In [ ]:
# Показываем топ-10 слов каждой темы
top_tokens = best_model_lda.score_tracker['top_tokens'].last_tokens

print(f'Топ-10 слов для каждой из {best_n} тем:')
print('=' * 60)
for topic_name, tokens in top_tokens.items():
    print(f'{topic_name}: {" | ".join(tokens)}')

In [ ]:
# Определяем доминирующую тему каждого тестового документа
dominant_topics = theta_test.values.argmax(axis=0)
topic_names = theta_test.index.tolist()

df_test_small['dominant_topic'] = dominant_topics
df_test_small['dominant_topic_name'] = [topic_names[i] for i in dominant_topics]

print('Распределение документов по темам (тестовая выборка):')
print(df_test_small['dominant_topic_name'].value_counts())

# Визуализация распределения
fig, ax = plt.subplots(figsize=(10, 4))
df_test_small['dominant_topic_name'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Тема', fontsize=12)
ax.set_ylabel('Число документов', fontsize=12)
ax.set_title(f'Распределение тестовых документов по {best_n} темам (LDA)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('test_topic_distribution.png', dpi=150)
plt.show()

In [ ]:
# Примеры документов для каждой темы
print('Примеры документов по темам:')
print('=' * 70)
for topic_idx, topic_name in enumerate(topic_names[:5]):  # первые 5 тем
    docs_in_topic = df_test_small[df_test_small['dominant_topic'] == topic_idx]
    print(f'\n[{topic_name}] ({len(docs_in_topic)} документов)')
    if len(docs_in_topic) > 0:
        sample_text = docs_in_topic.iloc[0]['text']
        print('  Пример:', sample_text[:200], '...')

## 7. Модель общего вида с регуляризаторами

In [ ]:
def build_regularized_model(n_topics, dictionary, batch_vectorizer_train,
                            tau_smooth_phi=0.1, tau_smooth_theta=0.1,
                            tau_sparse_phi=-0.05, tau_decorr=1e4,
                            n_passes=15, seed=42):
    """
    Модель общего вида с несколькими регуляризаторами:
      1. SmoothPhi     — сглаживание распределений тем по словам
      2. SmoothTheta   — сглаживание распределений тем в документах
      3. SparsePhi     — разреживание (усиление специфичности тем)
      4. DecorrelatorPhi — декорреляция тем (минимизация сходства тем)
    """
    model = artm.ARTM(
        num_topics=n_topics,
        dictionary=dictionary,
        num_document_passes=5,
        seed=seed,
        cache_theta=True
    )

    # --- Метрики ---
    model.scores.add(artm.PerplexityScore(name='perplexity', dictionary=dictionary))
    model.scores.add(artm.SparsityPhiScore(name='sparsity_phi'))
    model.scores.add(artm.SparsityThetaScore(name='sparsity_theta'))
    model.scores.add(artm.TopTokensScore(name='top_tokens', num_tokens=10))

    # --- Регуляризатор 1: SmoothSparsePhiRegularizer (phi — темы × слова) ---
    # Положительный tau → сглаживание; отрицательный → разреживание
    model.regularizers.add(artm.SmoothSparsePhiRegularizer(
        name='smooth_phi', tau=tau_smooth_phi
    ))

    # --- Регуляризатор 2: SmoothSparseThetaRegularizer (theta — темы × документы) ---
    model.regularizers.add(artm.SmoothSparseThetaRegularizer(
        name='smooth_theta', tau=tau_smooth_theta
    ))

    # --- Регуляризатор 3: Разреживание phi ---
    model.regularizers.add(artm.SmoothSparsePhiRegularizer(
        name='sparse_phi', tau=tau_sparse_phi
    ))

    # --- Регуляризатор 4: Декорреляция тем ---
    model.regularizers.add(artm.DecorrelatorPhiRegularizer(
        name='decorrelator_phi', tau=tau_decorr
    ))

    perplexity_values = []
    sparsity_phi_values = []
    sparsity_theta_values = []

    for i in range(n_passes):
        model.fit_offline(batch_vectorizer=batch_vectorizer_train, num_collection_passes=1)
        perp = model.score_tracker['perplexity'].last_value
        s_phi   = model.score_tracker['sparsity_phi'].last_value
        s_theta = model.score_tracker['sparsity_theta'].last_value
        perplexity_values.append(perp)
        sparsity_phi_values.append(s_phi)
        sparsity_theta_values.append(s_theta)
        print(f'  Итерация {i+1:2d}/{n_passes} | '
              f'Перплексия: {perp:.1f} | '
              f'Sparsity Phi: {s_phi:.3f} | '
              f'Sparsity Theta: {s_theta:.3f}')

    return model, perplexity_values, sparsity_phi_values, sparsity_theta_values


print(f'Обучаем регуляризованную модель с {best_n} темами...')
model_reg, perp_reg, sp_phi, sp_theta = build_regularized_model(
    n_topics=best_n,
    dictionary=dictionary,
    batch_vectorizer_train=batch_vectorizer_train,
    n_passes=20
)

In [ ]:
# График перплексии регуляризованной модели + сравнение с LDA
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Перплексия в ходе обучения
axes[0].plot(range(1, len(perp_reg)+1), perp_reg,
             marker='o', linewidth=2, color='tomato', label='Регуляризованная модель')
if len(perp_lda) > 0:
    # Для сравнения — LDA с тем же числом тем
    _, perp_lda_best, _, _ = build_regularized_model.__module__ and (None,), None, None, None
    # (уже посчитана выше для best_n, берём из results)
    perp_lda_cmp = results[best_n][1]
    axes[0].plot(range(1, len(perp_lda_cmp)+1), perp_lda_cmp,
                 marker='s', linewidth=2, color='steelblue', linestyle='--', label=f'LDA ({best_n} тем)')
axes[0].set_xlabel('Итерация', fontsize=12)
axes[0].set_ylabel('Перплексия', fontsize=12)
axes[0].set_title('Перплексия в ходе обучения: LDA vs Регуляризованная', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Разреженность phi и theta
axes[1].plot(range(1, len(sp_phi)+1), sp_phi,
             marker='o', linewidth=2, color='darkorange', label='Sparsity Phi')
axes[1].plot(range(1, len(sp_theta)+1), sp_theta,
             marker='s', linewidth=2, color='purple', linestyle='--', label='Sparsity Theta')
axes[1].set_xlabel('Итерация', fontsize=12)
axes[1].set_ylabel('Разреженность', fontsize=12)
axes[1].set_title('Разреженность матриц Phi и Theta', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('regularized_model_training.png', dpi=150)
plt.show()
print(f'Финальная перплексия регуляризованной модели (train): {perp_reg[-1]:.1f}')

## 8. Темы тестовой выборки и перплексия для регуляризованной модели

In [ ]:
# Добавляем перплексию для теста
model_reg.scores.add(artm.PerplexityScore(
    name='perplexity_test',
    dictionary=dictionary
))

# Трансформация тестовых данных
theta_test_reg = model_reg.transform(batch_vectorizer=batch_vectorizer_test)

test_perp_reg = model_reg.score_tracker['perplexity_test'].last_value
print(f'Перплексия на тестовой выборке (регуляризованная модель, {best_n} тем): {test_perp_reg:.1f}')
print(f'Перплексия на тестовой выборке (LDA,                       {best_n} тем): {test_perp:.1f}')

In [ ]:
# Топ-10 слов регуляризованной модели
top_tokens_reg = model_reg.score_tracker['top_tokens'].last_tokens

print(f'Топ-10 слов для каждой темы (регуляризованная модель):')
print('=' * 65)
for topic_name, tokens in top_tokens_reg.items():
    print(f'{topic_name}: {" | ".join(tokens)}')

In [ ]:
# Распределение документов по темам
dominant_topics_reg = theta_test_reg.values.argmax(axis=0)
topic_names_reg = theta_test_reg.index.tolist()

df_test_small['dominant_topic_reg'] = dominant_topics_reg
df_test_small['dominant_topic_reg_name'] = [topic_names_reg[i] for i in dominant_topics_reg]

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

df_test_small['dominant_topic_name'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title(f'LDA — темы тестовых документов', fontsize=13)
axes[0].set_xlabel('Тема'); axes[0].set_ylabel('Число документов')
axes[0].tick_params(axis='x', rotation=45)

df_test_small['dominant_topic_reg_name'].value_counts().plot(kind='bar', ax=axes[1], color='tomato')
axes[1].set_title(f'Регуляризованная — темы тестовых документов', fontsize=13)
axes[1].set_xlabel('Тема'); axes[1].set_ylabel('Число документов')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('comparison_test_distribution.png', dpi=150)
plt.show()

## 9. Итоговое сравнение

In [ ]:
print('=' * 60)
print('ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ')
print('=' * 60)
print(f'Оптимальное число тем (по перплексии):   {best_n}')
print()
print(f'LDA (BigARTM, Dirichlet-регуляризаторы):')
print(f'  Перплексия train (последняя итерация):  {results[best_n][0]:.1f}')
print(f'  Перплексия test:                        {test_perp:.1f}')
print()
print(f'Модель общего вида (4 регуляризатора):')
print(f'  Перплексия train (последняя итерация):  {perp_reg[-1]:.1f}')
print(f'  Перплексия test:                        {test_perp_reg:.1f}')
print(f'  Sparsity Phi  (финал):                  {sp_phi[-1]:.4f}')
print(f'  Sparsity Theta (финал):                 {sp_theta[-1]:.4f}')
print('=' * 60)

if test_perp_reg < test_perp:
    print('✅ Регуляризованная модель показывает ЛУЧШУЮ перплексию на тесте')
else:
    print('ℹ️  LDA показывает лучшую перплексию на тесте (возможно, стоит подобрать tau)')